In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from typing import Optional
import gymnasium as gym
from collections import deque


In [2]:
# Cada modelo lineal produce Q(s, a)
class LinearQ(nn.Module):
	def __init__(self, state_dim: int):
		super().__init__()
		self.linear = nn.Linear(state_dim, 1)

	def forward(self, state):
		return self.linear(state)
	

In [3]:
class ActionTreeNode:
	def __init__(self, state_dim: int, depth: int = 0, max_depth: int = 7, device: Optional[str] = None, buffer_size: int = 256):
		self.state_dim = state_dim
		self.depth = depth
		self.max_depth = max_depth
		self.device = torch.device(device) if device is not None else torch.device(
			"cuda" if torch.cuda.is_available() else "cpu"
		)
		self.buffer_size = buffer_size

		# Indica si el nodo es una hoja (modelo lineal) o es nodo que separa el código
		self.is_leaf = True
		self.model = LinearQ(state_dim).to(self.device)
		# Buffer con las experiencias
		self.buffer = deque(maxlen=buffer_size)

		self.split_feature = None
		self.split_threshold = None
		self.left: ActionTreeNode
		self.right: ActionTreeNode
		self.optimizer = None

	def route(self, state):
		if self.is_leaf:
			return self
		if state[self.split_feature] < self.split_threshold:
			return self.left.route(state)
		else:
			return self.right.route(state)
		
	def copy(self):
		new_node = ActionTreeNode(
			state_dim=self.state_dim,
			depth=self.depth,
			max_depth=self.max_depth,
			device=self.device,
			buffer_size=self.buffer_size,
		)
		new_node.is_leaf = self.is_leaf
		# No hace falta copiar el buffer, solo los pesos
		new_node.model.load_state_dict(self.model.state_dict())
		new_node.model.to(self.device)
		if not self.is_leaf:
			new_node.split_feature = self.split_feature
			new_node.split_threshold = self.split_threshold
			new_node.left = self.left.copy()
			new_node.right = self.right.copy()
		return new_node


In [ ]:
class AutonomousTreeAgent:
	def __init__(self, state_dim: int, n_actions: int, max_depth: int = 5, gamma: float = 0.99, lr: float = 1e-3, device: Optional[str] = None, target_update_freq: int = 10):
		self.state_dim = state_dim
		self.n_actions = n_actions
		self.gamma = gamma
		self.lr = lr
		self.device = torch.device(device) if device is not None else torch.device(
			"cuda" if torch.cuda.is_available() else "cpu"
		)

		self.target_update_freq = target_update_freq
		
		# Hay un árbol por cada acción
		self.trees: list[ActionTreeNode] = [
			ActionTreeNode(state_dim, max_depth=max_depth, device=self.device)
			for _ in range(n_actions)
		]

		self.target_trees = [tree.copy() for tree in self.trees]

		# El agente aprende con una política epsilon-greedy
		self.epsilon = 1.0
		self.epsilon_min = 0.01
		self.epsilon_decay = 0.995

	def _sync_target(self):
		self.target_trees = [tree.copy() for tree in self.trees]

	def q_values(self, state, use_target: bool = False):
		state_t = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)

		trees = self.target_trees if use_target else self.trees
		q_vals = []
		for a in range(self.n_actions):
			leaf = trees[a].route(state)
			with torch.no_grad():
				q = leaf.model(state_t).item()
			q_vals.append(q)
		return q_vals

	# Política epsilon-greedy
	def select_action(self, state):
		if random.random() < self.epsilon:
			return random.randint(0, self.n_actions - 1)
		q_vals = self.q_values(state)
		return int(np.argmax(q_vals))

	def store(self, state, action: int, reward: int, next_state, done: bool):
		# Encontrar el árbol que corresponde con la acción
		tree = self.trees[action]
		# Encontrar el modelo lineal que usar, que está en un nodo hoja
		leaf = tree.route(state)
		
		leaf.buffer.append((
			state, 
			action, 
			reward, 
			next_state, 
			done
		))

	def train_leaf(self, leaf: ActionTreeNode, batch_size: int = 64):
		if len(leaf.buffer) < batch_size:
			return float('inf')

		# Obtener un batch de experiencias
		batch = random.sample(leaf.buffer, batch_size)
		# Optimizador SGD
		if leaf.optimizer is None:
			leaf.optimizer = optim.Adam(leaf.model.parameters(), lr=self.lr)
		optimizer = leaf.optimizer

		total_loss = 0.0
		for state, action, reward, next_state, done in batch:
			state_t = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
			next_state_t = torch.tensor(next_state, dtype=torch.float32, device=self.device).unsqueeze(0)

			q_current = leaf.model(state_t).squeeze()

			with torch.no_grad():
				next_q_vals = []
				# Hay que pasar por todas las acciones porque ahora cada árbol corresponde a una acción
				for a_prime in range(self.n_actions):
					leaf_prime = self.target_trees[a_prime].route(next_state)
					# Se obtiene el valor de Q para el siguiente estado, probando con todas las acciones
					q_next = leaf_prime.model(next_state_t).item()
					next_q_vals.append(q_next)
				# Qtarget = 
				#   not terminal -> reward + gamma * max(Q(s, a'))
				#   terminal -> reward
				target = reward + (1 - done) * self.gamma * max(next_q_vals)

			target_t = torch.tensor(target, dtype=torch.float32, device=self.device)
			loss = (target_t - q_current) ** 2

			optimizer.zero_grad()
			loss.backward()
			optimizer.step()

			total_loss += loss.item()

		return total_loss / batch_size
	
	# "The final Variance criterion selects a split that generates child nodes whose Q values contain the least variance."
	def _variance_of_q(self, leaf: ActionTreeNode):
		if len(leaf.buffer) <= 0:
			return 0.0
		
		q_vals = []
		for state, _, _, _, _ in leaf.buffer:
			state_t = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)

			with torch.no_grad():
				q = leaf.model(state_t).item()
			# Guardar todos los valores q dadas todas las experiencias que hay en el buffer
			q_vals.append(q)
		return float(np.var(q_vals))

	def _predict_q(self, node: ActionTreeNode, state):
		state_t = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)

		with torch.no_grad():
			return node.model(state_t).item()

	def _try_split(self, node: ActionTreeNode, min_improvement: float = 0.01):
		# if node.depth >= node.max_depth:
			# return False
		if len(node.buffer) < 32:
			return False
		
		states = np.array([s for s, _, _, _, _ in node.buffer])
		# Se obtiene la varianza del nodo
		parent_var = self._variance_of_q(node)

		best_gain = -1.0
		best_feature = None
		best_thresh = None

		for feat in range(self.state_dim):
			feat_vals = states[:, feat]

			thresholds = np.percentile(feat_vals, [25, 50, 75])

			for t in thresholds:
				left_mask = feat_vals < t
				right_mask = feat_vals >= t

				left_indices = np.where(left_mask)[0]
				right_indices = np.where(right_mask)[0]

				if len(left_indices) < 4 or len(right_indices) < 4:
					continue

				# Obtener los valores Q de cada subconjunto
				left_q = [self._predict_q(node, states[i]) for i in left_indices]
				right_q = [self._predict_q(node, states[i]) for i in right_indices]

				# Obtener la varianza de cada subconjunto
				left_var = np.var(left_q)
				right_var = np.var(right_q)

				# Escalar por el número de experiencias en cada subconjunto
				w_left = len(left_q) / len(node.buffer)
				w_right = len(right_q) / len(node.buffer)

				# Varianza ponderada en base al número de experiencias que hay en cada subconjunto
				split_var = w_left * left_var + w_right * right_var

				gain = parent_var - split_var

				if gain > best_gain:
					best_gain = gain
					best_feature = feat
					best_thresh = t

		if best_gain > min_improvement:
			node.is_leaf = False
			node.split_feature = best_feature
			node.split_threshold = best_thresh

			node.left = ActionTreeNode(node.state_dim, node.depth + 1, node.max_depth, device=self.device)
			node.right = ActionTreeNode(node.state_dim, node.depth + 1, node.max_depth, device=self.device)

			node.left.model.load_state_dict(node.model.state_dict())
			node.right.model.load_state_dict(node.model.state_dict())
			
			for trans in node.buffer:
				s = trans[0]
				if s[best_feature] < best_thresh:
					node.left.buffer.append(trans)
				else:
					node.right.buffer.append(trans)

			node.buffer.clear()

			self.train_leaf(node.left, batch_size=min(32, len(node.left.buffer)))
			self.train_leaf(node.right, batch_size=min(32, len(node.right.buffer)))
			return True
		return False

	def update_all_leaves(self):
		def traverse(node: ActionTreeNode):
			if node.is_leaf:
				loss = self.train_leaf(node)
				split_occurred = False
				# if loss <= 0.05:
				split_occurred = self._try_split(node)
				
				if loss != float('inf'):
					returned_loss = loss
				else:
					returned_loss = 0.0

				if split_occurred:
					split_value = 1
				else:
					split_value = 0

				return 1, returned_loss, split_value
			else:
				left_leaf_count, left_loss, left_splits = traverse(node.left)
				right_leaf_count, right_loss, right_splits = traverse(node.right)
				leaf_count = left_leaf_count + right_leaf_count
				total_loss = left_loss + right_loss
				splits = left_splits + right_splits
				return leaf_count, total_loss, splits

		total_leaves = 0
		total_loss = 0.0
		total_splits = 0
		for tree in self.trees:
			leaf_count, loss_sum, splits = traverse(tree)
			total_leaves += leaf_count
			total_loss += loss_sum
			total_splits += splits

		avg_loss = total_loss / total_leaves if total_leaves > 0 else 0.0
		return {'leaf_count': total_leaves, 'avg_loss': avg_loss, 'splits': total_splits}

	def decay_epsilon(self):
		self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


In [5]:
def train_autonomous_tree(env, agent: AutonomousTreeAgent, n_episodes: int = 500):
	recent_rewards = []
		
	for episode in range(n_episodes):
		state, _ = env.reset()  
		done = False
		episode_reward = 0.0

		while not done:
			action = agent.select_action(state)
			
			next_state, reward, terminated, truncated, _ = env.step(action)
			done = terminated or truncated
			
			agent.store(state, action, reward, next_state, done)
			state = next_state
			episode_reward += reward

		stats = agent.update_all_leaves()
		agent.decay_epsilon()

		if (episode + 1) % agent.target_update_freq == 0:
			print("Copying weights")
			agent._sync_target()

		recent_rewards.append(episode_reward)

		if len(recent_rewards) > 100:
			recent_rewards.pop(0)

		avg_reward = sum(recent_rewards) / len(recent_rewards)

		print(f"Ep {episode+1:4d} | Reward: {episode_reward:6.1f} | "
			  f"Avg100: {avg_reward:7.2f} "
			  f"ε: {agent.epsilon:.3f} | Leaves: {stats['leaf_count']} | "
			  f"Splits: {stats['splits']} | AvgLoss: {stats['avg_loss']:.4f}")


In [6]:
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

print(state_dim, n_actions)


4 2


In [7]:
agent = AutonomousTreeAgent(state_dim, n_actions, max_depth=4)
print(f"Using device: {agent.device}")


Using device: cuda


In [8]:
train_autonomous_tree(env, agent, n_episodes=300)


Ep    1 | Reward:    9.0 | Avg100:    9.00 ε: 0.995 | Leaves: 2 | Splits: 0 | AvgLoss: 0.0000
Ep    2 | Reward:   36.0 | Avg100:   22.50 ε: 0.990 | Leaves: 2 | Splits: 0 | AvgLoss: 0.0000
Ep    3 | Reward:   15.0 | Avg100:   20.00 ε: 0.985 | Leaves: 2 | Splits: 0 | AvgLoss: 0.0000
Ep    4 | Reward:   11.0 | Avg100:   17.75 ε: 0.980 | Leaves: 2 | Splits: 1 | AvgLoss: 0.0000
Ep    5 | Reward:   16.0 | Avg100:   17.40 ε: 0.975 | Leaves: 3 | Splits: 1 | AvgLoss: 0.0000
Ep    6 | Reward:   15.0 | Avg100:   17.00 ε: 0.970 | Leaves: 4 | Splits: 1 | AvgLoss: 0.0000
Ep    7 | Reward:   25.0 | Avg100:   18.14 ε: 0.966 | Leaves: 5 | Splits: 1 | AvgLoss: 0.0000
Ep    8 | Reward:   14.0 | Avg100:   17.62 ε: 0.961 | Leaves: 6 | Splits: 0 | AvgLoss: 0.0000
Ep    9 | Reward:   22.0 | Avg100:   18.11 ε: 0.956 | Leaves: 6 | Splits: 0 | AvgLoss: 0.0000
Copying weights
Ep   10 | Reward:   20.0 | Avg100:   18.30 ε: 0.951 | Leaves: 6 | Splits: 2 | AvgLoss: 0.2702
Ep   11 | Reward:   10.0 | Avg100:   17.55 ε

KeyboardInterrupt: 

In [ ]:
def evaluate(agent: AutonomousTreeAgent, episodes: int = 10):
	total_reward = 0
	for _ in range(episodes):
		state, _ = env.reset()
		done = False
		while not done:
			q_vals = agent.q_values(state)
			action = int(np.argmax(q_vals))
			next_state, reward, terminated, truncated, _ = env.step(action)
			done = terminated or truncated
			total_reward += reward

	return total_reward / episodes

print(f"Average return: {evaluate(agent):.1f}")
